# Loan Analysis - Data Preprocessing

**Section Goals:**
> - Remove or fill any missing data.
> - Remove unnecessary or repetitive features.
> - Convert categorical string features to dummy variables.
> - Split data into train/test sets.
> - Remove outliers from training data.
> - Normalize/scale the features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

In [ ]:
data = pd.read_pickle("../data/processed/01_eda_output.pkl")
print(f"Data shape: {data.shape}")
data.head()

## Missing Values

In [ ]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

### `emp_title`

Realistically there are too many unique job titles to try to convert this to a dummy variable feature. Let's remove the `emp_title` column.

In [ ]:
print(f"Unique emp_title values: {data.emp_title.nunique()}")
data.drop('emp_title', axis=1, inplace=True)

### `emp_length`

Charge off rates are extremely similar across all employment lengths. So we are going to drop the `emp_length` column.

In [ ]:
data.emp_length.unique()

In [ ]:
for year in data.emp_length.dropna().unique():
    print(f"{year}:")
    vals = data[data.emp_length == year].loan_status.value_counts()
    print(f"  Charged Off rate: {(vals.get(0, 0) / vals.sum()) * 100:.2f}%")
    print()

In [ ]:
data.drop('emp_length', axis=1, inplace=True)

### `title`

The title column is simply a string subcategory/description of the purpose column. So we are going to drop the title column.

In [ ]:
data.drop('title', axis=1, inplace=True)

### `mort_acc`

There are many ways we could deal with this missing data. We will group the dataframe by the `total_acc` and calculate the mean of `mort_acc` per `total_acc` entry to fill in the missing values.

In [ ]:
print(f"Missing mort_acc: {data.mort_acc.isna().sum()}")
data.corr()['mort_acc'].drop('mort_acc').sort_values()

In [ ]:
total_acc_avg = data.groupby('total_acc')['mort_acc'].mean()

def fill_mort_acc(total_acc, mort_acc):
    if np.isnan(mort_acc):
        return total_acc_avg.get(total_acc, total_acc_avg.mean()).round()
    else:
        return mort_acc

data['mort_acc'] = data.apply(lambda x: fill_mort_acc(x['total_acc'], x['mort_acc']), axis=1)

### `revol_util` & `pub_rec_bankruptcies`

These two features have missing data points, but they account for less than 0.5% of the total data. So we are going to drop the rows with missing values.

In [ ]:
for column in data.columns:
    if data[column].isna().sum() != 0:
        missing = data[column].isna().sum()
        portion = (missing / data.shape[0]) * 100
        print(f"'{column}': number of missing values '{missing}' ==> {portion:.3f}%")

In [ ]:
data.dropna(inplace=True)
print(f"Shape after dropping NAs: {data.shape}")

## Categorical Variables and Dummy Variables

In [ ]:
print([column for column in data.columns if data[column].dtype == object])

### `term`

In [ ]:
data.term.unique()

In [ ]:
term_values = {' 36 months': 36, ' 60 months': 60}
data['term'] = data.term.map(term_values)
data.term.unique()

### `grade` & `sub_grade`

We know that `grade` is just a parent feature of `sub_grade`, so we are going to drop it.

In [ ]:
data.drop('grade', axis=1, inplace=True)

In [ ]:
dummies = ['sub_grade', 'verification_status', 'purpose', 'initial_list_status',
           'application_type', 'home_ownership']
data = pd.get_dummies(data, columns=dummies, drop_first=True)

### `zip_code`

The raw data already has `zip_code` as a separate column. We'll convert it to dummy variables.

In [ ]:
data['zip_code'].value_counts().head(10)

In [ ]:
data = pd.get_dummies(data, columns=['zip_code'], drop_first=True)

### `addr_state`

We'll drop `addr_state` since we already have `zip_code` dummies which capture geographic information.

In [ ]:
data.drop('addr_state', axis=1, inplace=True)

### `issue_d`

This would be data leakage — we wouldn't know beforehand whether or not a loan would be issued when using our model, so in theory we wouldn't have this feature. Let's drop it.

In [ ]:
data.drop('issue_d', axis=1, inplace=True)

### `earliest_cr_line`

This appears to be a historical timestamp feature. Extract the year from this feature.

In [ ]:
data['earliest_cr_line'] = data.earliest_cr_line.dt.year
print(f"Unique years: {data.earliest_cr_line.nunique()}")
data.earliest_cr_line.value_counts().head(10)

## Train Test Split

In [ ]:
w_p = data.loan_status.value_counts()[0] / data.shape[0]
w_n = data.loan_status.value_counts()[1] / data.shape[0]

print(f"Weight of positive values (Charged Off): {w_p:.4f}")
print(f"Weight of negative values (Fully Paid): {w_n:.4f}")

In [ ]:
train, test = train_test_split(data, test_size=0.33, random_state=42)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## Removing Outliers

We remove outliers only from the training set to avoid data leakage.

In [ ]:
print(f"Train shape before outlier removal: {train.shape}")
train = train[train['annual_inc'] <= 250000]
train = train[train['dti'] <= 50]
train = train[train['open_acc'] <= 40]
train = train[train['total_acc'] <= 80]
train = train[train['revol_util'] <= 120]
train = train[train['revol_bal'] <= 250000]
print(f"Train shape after outlier removal: {train.shape}")

## Normalizing the Data

In [ ]:
X_train, y_train = train.drop('loan_status', axis=1), train.loan_status
X_test, y_test = test.drop('loan_status', axis=1), test.loan_status

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Convert to float32 for model compatibility
X_train_scaled = np.array(X_train_scaled).astype(np.float32)
X_test_scaled = np.array(X_test_scaled).astype(np.float32)
y_train = np.array(y_train).astype(np.float32)
y_test = np.array(y_test).astype(np.float32)

print(f"X_train dtype: {X_train_scaled.dtype}")
print(f"y_train dtype: {y_train.dtype}")

In [ ]:
# Save processed data
np.savez("../data/processed/02_processed_data.npz",
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train, y_test=y_test)
joblib.dump(scaler, "../data/processed/02_scaler.joblib")

print("Saved processed data and scaler.")
print(f"Number of features: {X_train_scaled.shape[1]}")